In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import shutil
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features.feature_extraction import extract_all_features

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "IDRiD" / "B. Disease Grading"
IMAGE_DIR = DATA_DIR / "1. Original Images"
GROUNDTRUTH_DIR = DATA_DIR / "2. Groundtruths"

SPLITS = {
    "train": {
        "image_dir": IMAGE_DIR / "a. Training Set",
        "label_path": GROUNDTRUTH_DIR / "a. IDRiD_Disease Grading_Training Labels.csv",
        "output": "idrid_feature_matrix.csv",
        "expected_rows": 413,
    },
    "test": {
        "image_dir": IMAGE_DIR / "b. Testing Set",
        "label_path": GROUNDTRUTH_DIR / "b. IDRiD_Disease Grading_Testing Labels.csv",
        "output": "idrid_test_feature_matrix.csv",
        "expected_rows": 103,
    },
}

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for split, cfg in SPLITS.items():
    print(f"{split}: images exist={cfg['image_dir'].is_dir()}, labels exist={cfg['label_path'].is_file()}")

train: images exist=True, labels exist=True
test: images exist=True, labels exist=True


In [2]:
def build_feature_matrix(image_dir, label_path, split_name):
    # Labels (IDRiD headers have trailing spaces and empty extra columns)
    labels = pd.read_csv(label_path)
    labels.columns = labels.columns.str.strip()
    labels = labels[["Image name", "Retinopathy grade"]].dropna()
    labels = labels.rename(columns={"Image name": "image_id"})
    labels["image_id"] = labels["image_id"].astype(str).str.strip()
    labels["Retinopathy grade"] = labels["Retinopathy grade"].astype(int)

    image_paths = sorted(image_dir.glob("*.jpg"))
    print(f"[{split_name}] {len(image_paths)} images, {len(labels)} labels")

    rows, failed = [], []
    for i, image_path in enumerate(image_paths, start=1):
        try:
            features = extract_all_features(image_path, include_red_lesions=True)
        except Exception as e:
            failed.append((image_path.name, repr(e)))
            continue
        features["image_id"] = image_path.stem
        rows.append(features)
        if i % 25 == 0 or i == len(image_paths):
            print(f"  processed {i}/{len(image_paths)}")

    if failed:
        for name, err in failed:
            print("  FAILED:", name, err)
        raise RuntimeError(f"{len(failed)} images failed - CSV not built")

    features_df = pd.DataFrame(rows)

    # Every image must have a label and every label an image
    no_label = set(features_df["image_id"]) - set(labels["image_id"])
    no_image = set(labels["image_id"]) - set(features_df["image_id"])
    assert not no_label, f"Images without labels: {sorted(no_label)[:5]}"
    assert not no_image, f"Labels without images: {sorted(no_image)[:5]}"

    df = features_df.merge(labels, on="image_id", how="inner", validate="one_to_one")

    # Column order: image_id, features..., target
    feature_cols = [c for c in features_df.columns if c != "image_id"]
    return df[["image_id"] + feature_cols + ["Retinopathy grade"]]

In [3]:
train_cfg = SPLITS["train"]
train_df = build_feature_matrix(train_cfg["image_dir"], train_cfg["label_path"], "train")

assert len(train_df) == train_cfg["expected_rows"], len(train_df)
print("Train shape:", train_df.shape)   # expected (413, 38): 36 features + image_id + grade

train_df.to_csv(OUTPUT_DIR / train_cfg["output"], index=False)
print("Saved:", OUTPUT_DIR / train_cfg["output"])

[train] 413 images, 413 labels
  processed 25/413
  processed 50/413
  processed 75/413
  processed 100/413
  processed 125/413
  processed 150/413
  processed 175/413
  processed 200/413
  processed 225/413
  processed 250/413
  processed 275/413
  processed 300/413
  processed 325/413
  processed 350/413
  processed 375/413
  processed 400/413
  processed 413/413
Train shape: (413, 38)
Saved: /home/deepak/projects/aiml_biodata/data/processed/idrid_feature_matrix.csv


In [5]:
test_cfg = SPLITS["test"]
test_df = build_feature_matrix(test_cfg["image_dir"], test_cfg["label_path"], "test")

assert len(test_df) == test_cfg["expected_rows"], len(test_df)
assert list(test_df.columns) == list(train_df.columns), "Train/test columns differ"
print("Test shape:", test_df.shape)     # expected (103, 38)

test_df.to_csv(OUTPUT_DIR / test_cfg["output"], index=False)
print("Saved:", OUTPUT_DIR / test_cfg["output"])

[test] 103 images, 103 labels
  processed 25/103
  processed 50/103
  processed 75/103
  processed 100/103
  processed 103/103
Test shape: (103, 38)
Saved: /home/deepak/projects/aiml_biodata/data/processed/idrid_test_feature_matrix.csv


In [6]:
for cfg in SPLITS.values():
    df = pd.read_csv(OUTPUT_DIR / cfg["output"])
    print(f"{cfg['output']}: shape={df.shape}, missing values={int(df.isna().sum().sum())}")

# Quick look (training set only): do red-lesion counts tend to rise with grade?
red_cols = ["red_count", "red_small_count", "red_large_count", "red_quadrant_min_count"]
train_df.groupby("Retinopathy grade")[red_cols].median()

idrid_feature_matrix.csv: shape=(413, 38), missing values=0
idrid_test_feature_matrix.csv: shape=(103, 38), missing values=0


,red_count,red_small_count,red_large_count,red_quadrant_min_count
Retinopathy grade,,,,
0,146.5,143.0,3.0,25.0
1,128.0,126.0,3.0,21.5
2,150.0,144.0,5.0,25.5
3,150.5,142.5,9.0,26.5
4,131.0,115.0,10.0,18.0


In [9]:
df = pd.read_csv("../data/processed/idrid_feature_matrix.csv")

In [10]:
df.head()

,image_id,mean_r,mean_g,mean_b,std_r,std_g,std_b,median_r,median_g,median_b,...,red_count,red_small_count,red_large_count,red_total_area,red_area_ratio,red_mean_area,red_mean_contrast,red_quadrant_min_count,red_quadrants_over_thresh,Retinopathy grade
0,IDRiD_001,171.874544,84.761409,10.862817,28.550963,20.078228,4.606147,170.0,82.0,10.0,...,106,97,9,1959,0.004271,18.481132,3.456029,9,4,3
1,IDRiD_002,169.477023,83.419822,11.629892,26.866515,17.173646,4.661440,173.0,83.0,11.0,...,216,203,13,2691,0.005875,12.458333,3.351088,40,4,3
2,IDRiD_003,161.807197,47.938058,1.404106,30.105872,11.566063,1.461934,166.0,49.0,1.0,...,110,106,4,1401,0.003066,12.736364,3.222173,14,4,2
3,IDRiD_004,141.874462,76.912938,22.279137,35.989964,26.303989,8.630401,139.0,71.0,20.0,...,172,161,11,2541,0.005552,14.773256,3.309597,26,4,3
4,IDRiD_005,159.712094,85.984632,44.247228,23.333006,17.442928,12.035322,161.0,85.0,44.0,...,212,195,17,3676,0.008032,17.339623,3.278990,33,4,4


In [12]:
df.shape

(413, 38)